In [9]:
from dotenv import load_dotenv
import os
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from TabTransformer import run_experiment, infer_tabtransformer
from sklearn.metrics import precision_recall_curve, average_precision_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
#Set up .env file with Kaggle API Token
load_dotenv()
os.environ["KAGGLE_API_TOKEN"] = os.getenv("KAGGLE_API_TOKEN")
path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
file = os.path.join(path, 'creditcard.csv')
data = pd.read_csv(file)

print(f"# of Frauds = {np.sum(data['Class'])}")
print(np.sum(data['Class'])/len(data['Class'])*100, '%')

# of Frauds = 492
0.1727485630620034 %


In [20]:
#Split data into fraud and non-fraud to build high concentration of fraud in small training dataset
train_size = 2000
fraud_data = data[data['Class'] == 1].sample(frac=1, random_state=42)
non_fraud_data = data[data['Class'] == 0].sample(frac=1, random_state=42)
data_subset = pd.concat([fraud_data, non_fraud_data[:train_size - len(fraud_data)]]).sample(frac=1, random_state=42)

#Test model on small dataset
model, test_results, test_df = run_experiment(data_subset, lr=3e-3)
print(f'Test AUPRC: {test_results["auprc"]}')

Test AUPRC: 0.9553380314648697


In [12]:
#Two level grid search for optimal learning rate 
test_auprcs = []
lrs = [1e-5, 1e-4, 1e-3, 1e-2]
best_auprc = 0
best_lr = 0
for lr in lrs:
    for seed in range(3):
        data_subset = pd.concat([fraud_data, non_fraud_data.sample(n=train_size-len(fraud_data), random_state=seed)]).sample(frac=1, random_state=seed)
        model, test_results, test_df = run_experiment(data_subset, lr=lr)
        test_auprc = test_results['auprc']
        test_auprcs.append(test_auprc)
        if test_auprc > best_auprc:
            best_auprc = test_auprc
            best_lr = lr
    print(np.mean(np.array(test_auprcs)), lr)

print()
test_auprcs = []
lrs = [best_lr*.3, best_lr*0.7, best_lr*3, best_lr*7]
for lr in lrs:
    for seed in range(3):
        data_subset = pd.concat([fraud_data, non_fraud_data.sample(n=train_size-len(fraud_data), random_state=seed)]).sample(frac=1, random_state=seed)
        model, test_results, test_df = run_experiment(data_subset, lr=lr)
        test_auprc = test_results['auprc']
        test_auprcs.append(test_auprc)
    print(np.mean(np.array(test_auprcs)), lr)

0.5693455036395365 1e-05
0.7492775633780214 0.0001
0.8141500891028483 0.001
0.8473757526940133 0.01

0.9384904863213303 0.003
0.9422212127583083 0.006999999999999999
0.9477978164967377 0.03
0.9486376232818219 0.07


In [24]:
#Tabulate total parameters and training parameters to see model size
model, test_results, test_df = run_experiment(data_subset, lr=3e-3)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total params: {total_params:,}")
print(f"Trainable params: {trainable_params:,}")

Total params: 13,953
Trainable params: 13,953
